read data from bronze layer

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

In [0]:
races_df = spark.table(bronze_table)

keep only column require for analytics drop url column

In [0]:
from pyspark.sql import functions as F

selected_races_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingestion_date"),
    F.col("source_file")
)

standardise column with snaka case(countryName ==> country_name, data ---> race_data)

In [0]:
standardise_races_df = selected_races_df.withColumnsRenamed({
                                    "circuitId":"circuit_id",
                                    "date":"race_date",
                                    "raceName":"race_name",
                                })

#### Drop Duplicate Record

In [0]:
print(standardise_races_df.count())

In [0]:
races_distint_df = standardise_races_df.dropDuplicates(["season","round"]) # beacuase we have composite key

In [0]:
print(races_distint_df.count())

#### transform values of column race_name to title case

In [0]:
races_final_df = races_distint_df.withColumn("race_name",F.initcap(F.col("race_name")))
# display(races_final_df)

#### write data to silver layer

In [0]:
races_final_df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

In [0]:
display(spark.table(silver_table))